In [4]:
from torch.utils.data import Dataset
import pandas as pd
from pathlib import Path
import os 
from sklearn.preprocessing import LabelEncoder
from torchvision import transforms
from PIL import Image


class InsectDataset(Dataset):
    def __init__(self, dataset_path, image_path, transform=None):
        super().__init__()
        self.dataset_path = Path(dataset_path)
        self.image_path = image_path
        self.dataset = pd.DataFrame()
        self.transform = transform
        
        dfs = []
        for f in self.dataset_path.iterdir():
            if f.is_file() and f.suffix == '.csv':
                temp_df = pd.read_csv(f)
                temp_df['Filename'] = f.with_suffix('.jpg').name
                temp_df['image_path'] = os.path.join(image_path, f.with_suffix('.jpg').name)
                dfs.append(temp_df)
                
        if dfs:
            self.dataset = pd.concat(dfs, ignore_index=True)
            
        labels = LabelEncoder()
        if 'class_name' in self.dataset.columns:
            self.dataset['class_classification'] = labels.fit_transform(self.dataset['class_name'])
                
        
    def __len__(self):
        return self.dataset.shape[0]

    def __getitem__(self, idx):
        item = self.dataset.iloc[idx]
        image_path, bbox_x, bbox_y, bbox_w, bbox_h = item['image_path'], item['bbox_x'], item['bbox_y'], item['bbox_w'], item['bbox_h']
        
        image = Image.open(image_path).convert('RGB')
        
        cropped_image = transforms.functional.crop(image, top=int(bbox_y), left=int(bbox_x),
               height=int(bbox_h), width=int(bbox_w))
               
        if self.transform:
            img = self.transform(cropped_image)
        else:
            composer = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
            img = composer(cropped_image)
            
        return img, item['class_classification'], item['image_path']


In [21]:
img_path = r'E:\insects_proj\ShadowGraph\ShadowGraph\data\morphocluster-free-number-of-clusters-200imgs\200samples'
dataset_path = r'E:\insects_proj\ShadowGraph\ShadowGraph\data\200imgsResults_annotation_results\200imgsResults_annotation_results\200imgsResults_annotations_per_image'
dataset = InsectDataset(dataset_path, img_path)
print(len(dataset))
for i in range(10):
    img, label, path = dataset[i]
    print(f'Image Path: {path}, Label: {label}, Image Shape: {img.shape}')

9583
Image Path: E:\insects_proj\ShadowGraph\ShadowGraph\data\morphocluster-free-number-of-clusters-200imgs\200samples\RCNHSha_20250509_233918_IMG2.jpg, Label: 2, Image Shape: torch.Size([3, 84, 128])
Image Path: E:\insects_proj\ShadowGraph\ShadowGraph\data\morphocluster-free-number-of-clusters-200imgs\200samples\RCNHSha_20250509_233918_IMG2.jpg, Label: 2, Image Shape: torch.Size([3, 48, 40])
Image Path: E:\insects_proj\ShadowGraph\ShadowGraph\data\morphocluster-free-number-of-clusters-200imgs\200samples\RCNHSha_20250509_233918_IMG2.jpg, Label: 2, Image Shape: torch.Size([3, 66, 40])
Image Path: E:\insects_proj\ShadowGraph\ShadowGraph\data\morphocluster-free-number-of-clusters-200imgs\200samples\RCNHSha_20250509_233918_IMG2.jpg, Label: 2, Image Shape: torch.Size([3, 54, 37])
Image Path: E:\insects_proj\ShadowGraph\ShadowGraph\data\morphocluster-free-number-of-clusters-200imgs\200samples\RCNHSha_20250509_233918_IMG2.jpg, Label: 0, Image Shape: torch.Size([3, 57, 52])
Image Path: E:\inse

In [6]:
dataset.dataset.head()

,detection_id,bbox_x,bbox_y,bbox_w,bbox_h,original_cluster,final_class,class_name,status,Filename,image_path,class_classification
0,6,1749,2467,128,84,4,4,Detritus / Unsharp Organisms,confirmed,RCNHSha_20250509_233918_IMG2.jpg,E:\insects_proj\ShadowGraph\ShadowGraph\data\m...,2
1,4,3100,1894,40,48,7,7,Detritus / Unsharp Organisms,confirmed,RCNHSha_20250509_233918_IMG2.jpg,E:\insects_proj\ShadowGraph\ShadowGraph\data\m...,2
2,7,2092,2863,40,66,7,7,Detritus / Unsharp Organisms,confirmed,RCNHSha_20250509_233918_IMG2.jpg,E:\insects_proj\ShadowGraph\ShadowGraph\data\m...,2
3,0,1085,703,37,54,7,7,Detritus / Unsharp Organisms,confirmed,RCNHSha_20250509_233918_IMG2.jpg,E:\insects_proj\ShadowGraph\ShadowGraph\data\m...,2
4,5,2381,2341,52,57,14,0,Cladoceramorpha,relabelled,RCNHSha_20250509_233918_IMG2.jpg,E:\insects_proj\ShadowGraph\ShadowGraph\data\m...,0


In [7]:
import seaborn as sns
import matplotlib.pyplot as plt

count_values = dataset.dataset['class_classification'].value_counts()
count_values

class_classification
2    9077
0     420
1      67
6      12
5       4
3       2
4       1
Name: count, dtype: int64

Future directions:

1. Contrastive learning => learn embeddings in order to perform clustering(we can use triplet loss in order to make positive and anchor closer and everything else further apart)
2. Learning via Autoencoders => creating latents to perform clustering (? is it reasonable to use autoencoders in the terms of ML to get a compressed vector respresentation of the data)

In [7]:
import pytorch_lightning as pl
from torchmetrics import F1Score, Accuracy, Precision, Recall
from torch.utils.data import random_split, DataLoader
from torch import nn
import torch
import torchvision
import numpy as np

In [9]:
from torch.utils.data import Dataset
import pandas as pd
from pathlib import Path
import os 
from sklearn.preprocessing import LabelEncoder
from torchvision import transforms
from PIL import Image


class InsectDataset(Dataset):
    def __init__(self, dataset_path, image_path, transform=None):
        super().__init__()
        self.dataset_path = Path(dataset_path)
        self.image_path = image_path
        self.dataset = pd.DataFrame()
        self.transform = transform
        
        dfs = []
        for f in self.dataset_path.iterdir():
            if f.is_file() and f.suffix == '.csv':
                temp_df = pd.read_csv(f)
                temp_df['Filename'] = f.with_suffix('.jpg').name
                temp_df['image_path'] = os.path.join(image_path, f.with_suffix('.jpg').name)
                dfs.append(temp_df)
                
        if dfs:
            self.dataset = pd.concat(dfs, ignore_index=True)
            
        labels = LabelEncoder()
        if 'class_name' in self.dataset.columns:
            self.dataset['class_classification'] = labels.fit_transform(self.dataset['class_name'])
                

        condition = self.dataset['class_name'] != 'Detritus / Unsharp Organisms'
        self.dataset = self.dataset[condition]
        
    def __len__(self):
        return self.dataset.shape[0]

    def __getitem__(self, idx):
        item = self.dataset.iloc[idx]
        image_path, bbox_x, bbox_y, bbox_w, bbox_h = item['image_path'], item['bbox_x'], item['bbox_y'], item['bbox_w'], item['bbox_h']
        
        image = Image.open(image_path).convert('RGB')
        
        cropped_image = transforms.functional.crop(image, top=int(bbox_y), left=int(bbox_x),
               height=int(bbox_h), width=int(bbox_w))
               
        if self.transform:
            img = self.transform(cropped_image)
        else:
            composer = transforms.Compose([
                transforms.ToTensor(),
                transforms.Resize((299, 299)),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
            img = composer(cropped_image)
            
        return img, item['class_classification'], item['image_path']

In [3]:
class InsectDataModule(pl.LightningDataModule):
    def __init__(self, dataset_path, image_path, transform,batch_size):
        super().__init__()
        self.dataset_path = dataset_path
        self.image_path = image_path
        self.transform = transform
        self.batch_size = batch_size

    def setup(self, stage = None):
        self.dataset = InsectDataset(
            self.dataset_path, 
            self.image_path, 
            self.transform
        )
        self.train_val_dataset, self.test_dataset = random_split(
            self.dataset, [0.90, 0.10], generator=torch.Generator().manual_seed(42)
        )
        self.train_dataset, self.val_dataset = random_split(
            self.train_val_dataset, [0.80, 0.20]
        )
    
    def train_dataloader(self):
        return DataLoader(self.train_dataset, shuffle=True, batch_size = self.batch_size, num_workers = 0)
    
    def val_dataloader(self):
        return DataLoader(self.val_dataset, shuffle=False, batch_size = self.batch_size, num_workers = 0)
    
    def test_dataloader(self):
        return DataLoader(self.test_dataset, shuffle=False, batch_size = self.batch_size, num_workers = 0)

In [4]:
def image_to_tb(self, batch, batch_idx, step_name):
    tensorboard = self.logger.experiment
    if batch_idx % 200 != 0:
        return
    images = []
    for index, image in enumerate(batch[0]):
        image = transforms.Normalize(
            mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
            std=[1/0.229, 1/0.224, 1/0.225]
        )(image)
        images.append(image)
    if images:
        img_grid = torchvision.utils.make_grid(images)
        tensorboard.add_image('Batch images ' + step_name, img_grid, batch_idx)

        
def log_to_graph(self, value, var, name, global_step):
    self.logger.experiment.add_scalars(var, {name: value}, global_step)

In [12]:
class InsectsModel(pl.LightningModule):
    def __init__(self, task, num_classes, model, type, lr=1e-3, w2_decay=1e-5):
        super().__init__()
        self.f1_score_train = F1Score(task=task, num_classes=num_classes, average = 'none')
        self.recall_train = Recall(task = task, num_classes = num_classes, average = 'none')
        self.precision_train = Precision(task = task, num_classes = num_classes, average = 'none')
        self.accuracy_train = Accuracy(task = task, num_classes=num_classes, average='none')
        self.model_to_use = model
        
        self.f1_score_val = F1Score(task=task, num_classes=num_classes, average = 'none')
        self.recall_val = Recall(task = task, num_classes = num_classes, average = 'none')
        self.precision_val = Precision(task = task, num_classes = num_classes, average = 'none')
        self.accuracy_val = Accuracy(task = task, num_classes=num_classes, average='none')
        
        self.f1_score_test = F1Score(task=task, num_classes=num_classes, average = 'none')
        self.recall_test = Recall(task = task, num_classes = num_classes, average = 'none')
        self.precision_test = Precision(task = task, num_classes = num_classes, average = 'none')
        self.accuracy_test = Accuracy(task = task, num_classes=num_classes, average='none') 
        self.save_hyperparameters()
        self.loss = nn.CrossEntropyLoss()
        self.train_loss = []
        self.validation_loss = []
        self.test_loss = []

        if type == 'CNN_based':
            self.model = torch.hub.load('pytorch/vision:v0.10.0', 'inception_v3', pretrained=True)
            self.model.AuxLogits = None # Remove the auxiliary branch to force single output
            hidden_layers = self.model.fc.in_features
            self.model.fc = nn.Identity()
            self.classifier_head = nn.Linear(hidden_layers, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = self.classifier_head(x)
        return x
        
    def training_step(self, batch):
        features, target, img_path = batch
        pred = self(features)
        loss = self.loss(pred, target)

        self.f1_score_train(pred, target)
        self.recall_train(pred, target)
        self.precision_train(pred, target)
        self.accuracy_train(pred, target)

        self.train_loss.append(loss.item())

        image_to_tb(self, batch, self.current_epoch, 'train')

        return loss
        
    def on_train_epoch_end(self):
        name = 'train'

        log_to_graph(self, np.mean(self.train_loss),           'loss',      name, self.current_epoch)

        accuracy_per_class = self.accuracy_train.compute()
        for i, class_accuracy in enumerate(accuracy_per_class):
            log_to_graph(self, class_accuracy.item(), f'accuracy_score_class_{i}', name, self.current_epoch)

        precision_per_class = self.precision_train.compute()
        for i, class_precision in enumerate(precision_per_class):
            log_to_graph(self, class_precision.item(), f'precision_score_class_{i}', name, self.current_epoch)

        f1_per_class = self.f1_score_train.compute()
        for i, class_f1 in enumerate(f1_per_class):
            log_to_graph(self, class_f1.item(), f'f1_score_class_{i}', name, self.current_epoch)


        recall_per_class = self.recall_train.compute()
                
        for i, class_recall in enumerate(recall_per_class):
            log_to_graph(self, class_recall.item(), f'recall_score_class_{i}', name, self.current_epoch)


        self.train_loss = []
        self.accuracy_train.reset()
        self.f1_score_train.reset()
        self.precision_train.reset()
        self.recall_train.reset()

    def validation_step(self, batch):
        features, target, img_path = batch
        pred = self(features)
        loss = self.loss(pred, target)

        self.f1_score_val(pred, target)
        self.recall_val(pred, target)
        self.precision_val(pred, target)
        self.accuracy_val(pred, target)

        self.validation_loss.append(loss.item())

        image_to_tb(self, batch, self.current_epoch, 'val')

        return loss
    
    def on_validation_epoch_end(self):
        name = 'val'

        log_to_graph(self, np.mean(self.validation_loss), 'loss', name, self.current_epoch)

        accuracy_per_class = self.accuracy_val.compute()
        for i, class_accuracy in enumerate(accuracy_per_class):
            log_to_graph(self, class_accuracy.item(), f'accuracy_class_{i}', name, self.current_epoch)

        precision_per_class = self.precision_val.compute()
        for i, class_precision in enumerate(precision_per_class):
            log_to_graph(self, class_precision.item(), f'precision_class_{i}', name, self.current_epoch)

        f1_per_class = self.f1_score_val.compute()
        for i, class_f1 in enumerate(f1_per_class):
            log_to_graph(self, class_f1.item(), f'f1_score_class_{i}', name, self.current_epoch)

        recall_per_class = self.recall_val.compute()
        for i, class_recall in enumerate(recall_per_class):
            log_to_graph(self, class_recall.item(), f'recall_class_{i}', name, self.current_epoch)

        self.validation_loss = []
        self.accuracy_val.reset()
        self.f1_score_val.reset()
        self.precision_val.reset()
        self.recall_val.reset()

    def test_step(self, batch):
        features, target, img_path = batch
        pred = self(features)
        loss = self.loss(pred, target)

        self.f1_score_test(pred, target)
        self.recall_test(pred, target)
        self.precision_test(pred, target)
        self.accuracy_test(pred, target)

        self.test_loss.append(loss.item())

        image_to_tb(self, batch, self.current_epoch, 'test')

        return loss
    
    def on_test_epoch_end(self):
        name = 'test'

        log_to_graph(self, np.mean(self.test_loss), 'loss', name, self.current_epoch)

        accuracy_per_class = self.accuracy_test.compute()
        for i, class_accuracy in enumerate(accuracy_per_class):
            log_to_graph(self, class_accuracy.item(), f'accuracy_class_{i}', name, self.current_epoch)

        precision_per_class = self.precision_test.compute()
        for i, class_precision in enumerate(precision_per_class):
            log_to_graph(self, class_precision.item(), f'precision_class_{i}', name, self.current_epoch)

        f1_per_class = self.f1_score_test.compute()
        for i, class_f1 in enumerate(f1_per_class):
            log_to_graph(self, class_f1.item(), f'f1_score_class_{i}', name, self.current_epoch)

        recall_per_class = self.recall_test.compute()
        for i, class_recall in enumerate(recall_per_class):
            log_to_graph(self, class_recall.item(), f'recall_class_{i}', name, self.current_epoch)

        self.test_loss = []
        self.accuracy_test.reset()
        self.f1_score_test.reset()
        self.precision_test.reset()
        self.recall_test.reset()

    def configure_optimizers(self):
        parameters_model = [p for p in self.model.parameters() if p.requires_grad is True]
        parameters_clh = [p for p in self.classifier_head.parameters() if p.requires_grad is True]
        parameters = parameters_model + parameters_clh
        optimizer = torch.optim.AdamW(parameters, lr=self.hparams.lr, weight_decay=self.hparams.w2_decay)
        return optimizer

In [13]:
from pytorch_lightning.loggers import TensorBoardLogger

tb_logger = TensorBoardLogger(save_dir=r'E:\insects_proj\ShadowGraph\ShadowGraph\output_dir', name='lightning_logs')

trainer = pl.Trainer(
        accelerator='auto',
        logger=tb_logger,
        default_root_dir=r'E:\insects_proj\ShadowGraph\ShadowGraph\output_dir',
        enable_checkpointing=True,
        max_epochs=10,
        precision=16,
        log_every_n_steps=50,
    )
img_path = r'E:\insects_proj\ShadowGraph\ShadowGraph\data\morphocluster-free-number-of-clusters-200imgs\200samples'
dataset_path = r'E:\insects_proj\ShadowGraph\ShadowGraph\data\200imgsResults_annotation_results\200imgsResults_annotation_results\200imgsResults_annotations_per_image'

datamodule = InsectDataModule(dataset_path, img_path, transform = None, batch_size = 64)
num_classes = 6
model = InsectsModel(task='multiclass', num_classes=num_classes, type='CNN_based', model='VGG', lr=0.001, w2_decay=0.01)
trainer.fit(datamodule=datamodule, model = model)

Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Using cache found in C:\Users\mrgan/.cache\torch\hub\pytorch_vision_v0.10.0
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
e:\insects_proj\.venv\Lib\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

   | Name            | Type                | Params | Mode  | FLOPs
-------------------------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

e:\insects_proj\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
e:\insects_proj\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

IndexError: Target 6 is out of bounds.